# 05 Validation Diagnostics

## Purpose

This notebook reviews one finished training run using the saved trainer history.

## Inputs

- `trainer_state.json` from one experiment folder in `MyDrive/ProjectRoot2/checkpoints/`
- `experiment_metadata.json` from that same experiment folder
- `MyDrive/ProjectRoot2/registry/run_index.csv`

## Outputs

- validation loss plot by epoch
- exported loss-history CSV
- `validation_summary.json`
- updated run-index entry with the validation summary path

## Notes to myself

This notebook is only for training diagnostics. I am using it to decide whether a run looks finished, whether the best validation point came late, and whether a continuation run is worth trying.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- validation outputs stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

# Mount Drive so this runtime can read checkpoints and save validation outputs.
drive.mount('/content/drive')

# Define the private GitHub repo used for the rebuilt workflow.
GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta-sandbox2'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

# Clone the repo into the runtime if needed. Otherwise update the existing clone.
if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

# Add the repo to sys.path so imports from src/ work in Colab.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

## Select the training run

The only things I should need to change here are the tokenizer family and the experiment name. Everything else gets built from those two fields.

In [ ]:
# ==============================================================================
# 1. DEFINE THE RUN TO REVIEW
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'

TOKENIZER_FAMILY = 'hybrid_char_bpe'   # 'byte_bpe', 'manual', or 'hybrid_char_bpe'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv70_m2'

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', TOKENIZER_FAMILY, EXPERIMENT_NAME)
TRAINER_STATE_PATH = os.path.join(CHECKPOINT_DIR, 'trainer_state.json')
EXPERIMENT_METADATA_PATH = os.path.join(CHECKPOINT_DIR, 'experiment_metadata.json')
RUN_INDEX_PATH = os.path.join(PROJECT_ROOT, 'registry', 'run_index.csv')
VALIDATION_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'validation', TOKENIZER_FAMILY, EXPERIMENT_NAME)

os.makedirs(VALIDATION_RESULTS_DIR, exist_ok=True)

for required_path in [CHECKPOINT_DIR, TRAINER_STATE_PATH, EXPERIMENT_METADATA_PATH, RUN_INDEX_PATH]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Required path not found: {required_path}')

print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Validation results directory: {VALIDATION_RESULTS_DIR}')

## Load the training history and metadata

This is the core validation-diagnostics input: the saved trainer log history. I also want the experiment metadata so the summary file keeps the run context attached.

In [ ]:
# ==============================================================================
# 2. LOAD TRAINER HISTORY AND EXPERIMENT METADATA
# ==============================================================================
import json

from src.training_diagnostics import load_trainer_history, split_train_eval_history

with open(EXPERIMENT_METADATA_PATH, 'r', encoding='utf-8') as file:
    experiment_metadata = json.load(file)

log_history_df = load_trainer_history(TRAINER_STATE_PATH)
train_rows, eval_rows = split_train_eval_history(log_history_df)

if train_rows.empty or eval_rows.empty:
    raise ValueError('Trainer history does not contain both training and validation loss rows.')

display(train_rows.head())
display(eval_rows.head())


## Plot training and validation loss

This is the main plot I care about here. I want epochs on the x-axis and loss on the y-axis so I can see whether validation improved late, flattened out, or turned upward.

In [ ]:
# ==============================================================================
# 3. PLOT AND SAVE THE LOSS CURVES
# ==============================================================================
from src.training_diagnostics import plot_loss_curves, save_loss_curve_plot

plot_loss_curves(train_rows, eval_rows)

loss_plot_path = os.path.join(VALIDATION_RESULTS_DIR, 'loss_curves.png')
save_loss_curve_plot(train_rows, eval_rows, loss_plot_path)

print(f'Loss plot saved to: {loss_plot_path}')

## Summarize the run

I want a compact summary here instead of just a plot. The key questions are:

- what epoch gave the best validation loss
- what the last validation loss was
- whether the best point came late enough that continuation might still make sense

In [ ]:
# ==============================================================================
# 4. BUILD THE VALIDATION SUMMARY
# ==============================================================================
import pandas as pd

from src.training_diagnostics import merge_loss_history, recommend_continuation, summarize_best_epoch

loss_history_export = merge_loss_history(train_rows, eval_rows)
loss_history_path = os.path.join(VALIDATION_RESULTS_DIR, 'loss_history.csv')
loss_history_export.to_csv(loss_history_path, index=False)

best_epoch_summary = summarize_best_epoch(eval_rows)
total_epochs = int(float(experiment_metadata['live_hyperparameters']['epochs']))
continuation_recommendation = recommend_continuation(eval_rows, total_epochs=total_epochs)

validation_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'tokenizer_family': experiment_metadata['live_hyperparameters'].get('tokenizer_family', TOKENIZER_FAMILY),
    'setting_label': experiment_metadata['live_hyperparameters'].get('setting_label', ''),
    'run_mode': experiment_metadata['live_hyperparameters'].get('run_mode', ''),
    'checkpoint_dir': CHECKPOINT_DIR,
    'trainer_state_path': TRAINER_STATE_PATH,
    'loss_plot_path': loss_plot_path,
    'loss_history_path': loss_history_path,
    'best_epoch_summary': best_epoch_summary,
    'continuation_recommendation': continuation_recommendation,
}

summary_df = pd.DataFrame(
    {
        'metric': [
            'best_epoch',
            'best_val_loss',
            'last_epoch',
            'last_val_loss',
            'continuation_recommendation',
        ],
        'value': [
            best_epoch_summary['best_epoch'],
            best_epoch_summary['best_val_loss'],
            best_epoch_summary['last_epoch'],
            best_epoch_summary['last_val_loss'],
            continuation_recommendation,
        ],
    }
)

display(summary_df)

## Save the validation summary and update the run index

I want the summary saved into the validation results folder and also linked from the run index so I can find it again later without hunting through Drive.

In [ ]:
# ==============================================================================
# 5. SAVE THE SUMMARY AND REGISTER IT IN THE RUN INDEX
# ==============================================================================
from src.run_index import upsert_run_record

validation_summary_path = os.path.join(VALIDATION_RESULTS_DIR, 'validation_summary.json')
with open(validation_summary_path, 'w', encoding='utf-8') as file:
    json.dump(validation_summary, file, indent=2)

upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': validation_summary['tokenizer_family'],
        'setting_label': validation_summary['setting_label'],
        'run_mode': validation_summary['run_mode'],
        'parent_experiment_name': experiment_metadata['live_hyperparameters'].get('parent_experiment_name', ''),
        'mlm_probability': experiment_metadata['live_hyperparameters'].get('mlm_probability', ''),
        'num_hidden_layers': experiment_metadata['live_hyperparameters'].get('num_hidden_layers', ''),
        'attention_heads': experiment_metadata['live_hyperparameters'].get('attention_heads', ''),
        'hidden_size': experiment_metadata['live_hyperparameters'].get('hidden_size', ''),
        'intermediate_size': experiment_metadata['live_hyperparameters'].get('intermediate_size', ''),
        'batch_size': experiment_metadata['live_hyperparameters'].get('batch_size', ''),
        'learning_rate': experiment_metadata['live_hyperparameters'].get('learning_rate', ''),
        'weight_decay': experiment_metadata['live_hyperparameters'].get('weight_decay', ''),
        'epochs': experiment_metadata['live_hyperparameters'].get('epochs', ''),
        'early_stopping_patience': experiment_metadata['live_hyperparameters'].get('early_stopping_patience', ''),
        'tokenizer_dir': experiment_metadata['vault_routing'].get('tokenizer_dir', ''),
        'tokenized_dataset_dir': experiment_metadata['vault_routing'].get('tokenized_dataset_dir', ''),
        'checkpoint_dir': experiment_metadata['vault_routing'].get('checkpoint_dir', CHECKPOINT_DIR),
        'results_dir': VALIDATION_RESULTS_DIR,
        'validation_summary_path': validation_summary_path,
        'notebook_used': 'notebooks/05_validation_diagnostics2.ipynb',
        'git_commit': experiment_metadata.get('git_commit', ''),
        'run_status': experiment_metadata.get('run_status', ''),
        'notes': continuation_recommendation,
    },
)

print(f'Validation summary saved to: {validation_summary_path}')
print(f'Loss history CSV saved to: {loss_history_path}')

## GitHub sync note

Same idea as the earlier notebooks. The validation outputs stay in Drive. The notebook itself stays versioned in GitHub.

In [ ]:
# ==============================================================================
# 6. SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks', '05_validation_diagnostics2.ipynb')
DRIVE_NOTEBOOK_PATH = '/content/drive/MyDrive/Colab Notebooks/05_validation_diagnostics2.ipynb'

if os.path.exists(DRIVE_NOTEBOOK_PATH):
    !cp "{DRIVE_NOTEBOOK_PATH}" "{REPO_NOTEBOOK_PATH}"

    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/05_validation_diagnostics2.ipynb src/training_diagnostics.py
    !git commit -m "Update 05_validation_diagnostics2" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
else:
    print(f'Notebook file not found at: {DRIVE_NOTEBOOK_PATH}')
    print('Save the notebook in Colab, then run this cell again.')